# Civic PDF Parsing Findings

This notebook summarizes parser experiment outputs and provides manual scoring columns for document-structure quality.

In [ ]:
from pathlib import Path
import json
import pandas as pd

OUT_DIR_CANDIDATES = [Path('out'), Path('experiments/out')]
OUT_DIR = next((path for path in OUT_DIR_CANDIDATES if path.exists()), OUT_DIR_CANDIDATES[0])

rows = []
for metadata_path in sorted(OUT_DIR.glob('*/**/metadata.json')):
    rows.append(json.loads(metadata_path.read_text()))

df = pd.DataFrame(rows)
if df.empty:
    df = pd.DataFrame(columns=[
        'parser', 'filename', 'status', 'runtime_seconds', 'bytes',
        'markdown_path', 'json_path', 'error'
    ])

score_columns = [
    'reading_order_score',
    'header_footer_score',
    'attendance_score',
    'agenda_item_score',
    'motion_vote_score',
    'extraction_readiness_score',
    'manual_notes',
]
for column in score_columns:
    if column not in df.columns:
        df[column] = ''

display_columns = [
    'parser', 'filename', 'status', 'runtime_seconds', 'bytes',
    'reading_order_score', 'header_footer_score', 'attendance_score',
    'agenda_item_score', 'motion_vote_score', 'extraction_readiness_score',
    'manual_notes', 'markdown_path', 'json_path', 'error',
]
df[display_columns]

## Keyword Probes

These quick probes are not extractors. They help identify whether the parsed Markdown keeps civic concepts visible enough for later structured extraction.

In [ ]:
keywords = ['present', 'absent', 'motion', 'second', 'aye', 'yes', 'no', 'abstain', 'unanimous']
probe_rows = []

for _, row in df.iterrows():
    md_path = Path(row.get('markdown_path', ''))
    if not md_path.exists():
        continue
    text = md_path.read_text(encoding='utf-8').lower()
    probe_rows.append({
        'parser': row.get('parser'),
        'filename': row.get('filename'),
        **{keyword: text.count(keyword) for keyword in keywords},
    })

pd.DataFrame(probe_rows)

## Hybrid Selective LLM

This experiment uses Docling Standard output as the fast extraction layer, then selects only high-value sections for possible LLM parsing. The default run is a dry run: it writes prompts and handoff metadata without calling a model.


In [ ]:
hybrid_dir = OUT_DIR / 'hybrid_selective_llm'
hybrid_summary_path = hybrid_dir / 'summary.json'
if hybrid_summary_path.exists():
    hybrid_summary = json.loads(hybrid_summary_path.read_text())
    display(pd.DataFrame([hybrid_summary]))
else:
    print('No hybrid summary found. Run experiments/run_hybrid_selective_llm.py first.')


In [ ]:
scores_path = hybrid_dir / 'section_scores.json'
if scores_path.exists():
    scores = pd.DataFrame(json.loads(scores_path.read_text()))
    handoffs = scores[scores['prompt_path'].notna()].copy()
    display(
        handoffs.sort_values(['filename', 'section_id'])[[
            'filename', 'doc_type', 'item_number', 'handoff_type',
            'word_count', 'ocr_joined_tokens', 'reasons', 'title', 'prompt_path'
        ]].reset_index(drop=True)
    )
else:
    print('No hybrid section scores found.')


In [ ]:
if scores_path.exists():
    display(
        handoffs.groupby(['doc_type', 'handoff_type'])
        .size()
        .rename('sections')
        .reset_index()
        .sort_values(['doc_type', 'sections'], ascending=[True, False])
    )
